# OntologyRAG-Q on Kaggle — a runnable RAG pipeline

This notebook is a **from-scratch, runnable implementation** inspired by the paper
*"OntologyRAG-Q: Resource Development and Benchmarking for Retrieval-Augmented
Question Answering in Qur'anic Tafsir"* (EMNLP 2025) and its resource repo:
https://github.com/sazani/OntologyRAG-Q

**Important:** the source repo only publishes *data* (the ontology-annotated
QA dataset, Tafsir books, and benchmark CSVs) — it does not publish the paper's
pipeline code. Everything executed below (chunking, retrieval, generation,
evaluation) is a simplified reimplementation built from the dataset and the
paper's description, not the authors' original code.

## What this notebook does
1. Downloads the public dataset (`OntologyQA_v1.json` + benchmark CSVs) directly
   from GitHub.
2. Builds **verse-level, ontology-tagged chunks** ("Ayat-Ontology chunking"):
   each chunk = one Qur'an verse's Tafsir commentary, tagged with the question
   *type* categories (the ontology) seen for that verse.
3. Embeds the chunks with a multilingual sentence embedding model and indexes
   them with FAISS (the *retrieval* half of RAG).
4. Answers a question by retrieving the most relevant chunk(s) and generating
   an answer grounded in that context with a **free, open-weight LLM**
   (`Qwen2.5-1.5B-Instruct` — runs on Kaggle's free T4 GPU). An optional
   OpenAI GPT branch is included if you want higher-quality answers and have
   an API key.
5. Evaluates generated answers against the gold answers with a lexical
   token-F1 score, plus an optional BERTScore (the metric the paper found
   correlated best with human judgment).

## How to run on Kaggle
1. Go to https://www.kaggle.com/code -> **New Notebook**.
2. Notebook Settings (right sidebar) -> **Accelerator: GPU T4 x1**,
   **Internet: On** (required to download the dataset and models).
3. Upload this file (`File -> Import Notebook`) or copy/paste the cells.
4. (Optional, for higher-quality answers) Add your OpenAI key as a Kaggle
   secret named `OPENAI_API_KEY` via `Add-ons -> Secrets`.
5. `Run All`.


In [ ]:
# 1. Install dependencies (Kaggle already ships torch/transformers, this just
# pins the extra libraries this notebook needs).
!pip install -q sentence-transformers faiss-cpu bert-score


In [ ]:
# 2. Download the public OntologyRAG-Q dataset directly from GitHub.
import os, urllib.request

DATA_DIR = "ontologyrag_q_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE = "https://raw.githubusercontent.com/sazani/OntologyRAG-Q/main/Resources"
FILES = {
    "OntologyQA_v1.json": f"{BASE}/OntologyQA_v1.json",
    "tafsir-questions.csv": f"{BASE}/Benchmark/tafsir-questions.csv",
    "source_ayserAttafaser.csv": f"{BASE}/Benchmark/source_ayserAttafaser.csv",
}

for name, url in FILES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", name)
        urllib.request.urlretrieve(url, path)
    else:
        print("Already downloaded", name)


## 3. Load the QA dataset

`OntologyQA_v1.json` already links each question to:
- the Qur'an verse (`Sura_ID`, `SURA_name`, `Verse_Number_start/End`, `Ayah_text`)
- the Tafsir commentary for that verse (`Tafsir_text`, `Source_name`)
- the gold `Question` / `Answer` pair
- **ontology labels** for the question (`Q_Type1`, `Q_type2_final_English`,
  `Q_type2_Arabic`) — this is the "ontology" side of Ayat-Ontology chunking.


In [ ]:
import json
import pandas as pd

with open(os.path.join(DATA_DIR, "OntologyQA_v1.json"), encoding="utf-8") as f:
    qa_records = json.load(f)

qa_df = pd.DataFrame(qa_records)
print(qa_df.shape)
qa_df[["SURA_name", "Ayah_text", "Question", "Answer", "Q_Type1", "Q_type2_final_English"]].head(5)


## 4. Build Ayat-Ontology chunks

Multiple questions share the same verse/Tafsir passage, so we deduplicate down
to one chunk per verse (per source), and attach the set of ontology categories
that were asked about that verse. This mirrors the paper's idea of chunking
*at the verse level* and tagging each chunk with ontology metadata, instead of
chunking the raw text blindly by character/token count.


In [ ]:
group_cols = ["Sura_ID", "SURA_name", "Verse_Number_start", "Verse_Number_End", "Source_name"]

# A small number of rows (~234 of 4199) are missing verse/Tafsir metadata entirely
# (no Sura_ID, no Tafsir_text) -- drop those before chunking, they can't be grounded.
qa_df_clean = qa_df.dropna(subset=group_cols + ["Tafsir_text"]).copy()
print(f"Dropped {len(qa_df) - len(qa_df_clean)} rows with missing verse/Tafsir metadata")

chunks = (
    qa_df_clean.groupby(group_cols)
    .agg(
        ayah_text=("Ayah_text", "first"),
        tafsir_text=("Tafsir_text", "first"),
        ontology_types=("Q_Type1", lambda s: sorted(set(s.dropna()))),
        ontology_categories=("Q_type2_final_English", lambda s: sorted(set(s.dropna()))),
    )
    .reset_index()
)
chunks["chunk_id"] = chunks.index
print("Unique Ayat-Ontology chunks:", len(chunks))
chunks.head(3)


## 5. Embed the chunks and build a FAISS retrieval index

Using a multilingual sentence embedding model since the corpus is Arabic.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Embed verse text + commentary together so retrieval matches on both.
corpus_texts = (chunks["ayah_text"] + " " + chunks["tafsir_text"]).tolist()
corpus_embeddings = embedder.encode(
    corpus_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(np.asarray(corpus_embeddings, dtype="float32"))
print("Indexed", index.ntotal, "chunks")


In [ ]:
def retrieve(question, top_k=3):
    q_emb = embedder.encode([question], normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q_emb, dtype="float32"), top_k)
    results = chunks.iloc[idxs[0]].copy()
    results["score"] = scores[0]
    return results

# quick sanity check
retrieve("ما هي البسملة؟", top_k=3)[["SURA_name", "ayah_text", "score"]]


## 6. Generate grounded answers (free, open-weight LLM by default)

`Qwen2.5-1.5B-Instruct` is small enough to run on Kaggle's free T4 GPU and has
solid Arabic support. If you added an `OPENAI_API_KEY` Kaggle secret, this
notebook will instead use GPT for generation (closer to the paper's
best-performing model).


In [ ]:
USE_OPENAI = False
try:
    from kaggle_secrets import UserSecretsClient
    OPENAI_API_KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
    USE_OPENAI = bool(OPENAI_API_KEY)
except Exception:
    OPENAI_API_KEY = None

print("Using OpenAI GPT for generation" if USE_OPENAI else "Using free local LLM (Qwen2.5-1.5B-Instruct) for generation")

if USE_OPENAI:
    !pip install -q openai
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def generate_answer(question, context):
        prompt = (
            "بناءً على النص التالي من التفسير، أجب عن السؤال بإيجاز ودقة "
            "معتمدًا فقط على المعلومات الواردة في النص.\n\n"
            f"النص:\n{context}\n\nالسؤال: {question}\nالإجابة:"
        )
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return resp.choices[0].message.content.strip()
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    gen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
    gen_model = AutoModelForCausalLM.from_pretrained(
        gen_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

    def generate_answer(question, context):
        prompt = (
            "بناءً على النص التالي من التفسير، أجب عن السؤال بإيجاز ودقة "
            "معتمدًا فقط على المعلومات الواردة في النص.\n\n"
            f"النص:\n{context}\n\nالسؤال: {question}"
        )
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(gen_model.device)
        output = gen_model.generate(**inputs, max_new_tokens=200, do_sample=False)
        answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return answer.strip()


## 7. Put it together: the full RAG answer function

In [ ]:
def answer_question(question, top_k=3, verbose=True):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n---\n".join(retrieved["tafsir_text"].tolist())
    answer = generate_answer(question, context)
    if verbose:
        print("Q:", question)
        print("Retrieved verses:", retrieved["SURA_name"].tolist(), retrieved["Verse_Number_start"].tolist())
        print("Generated answer:", answer)
    return answer, retrieved

_ = answer_question("ما معنى الرحمن؟")


## 8. Evaluate against the benchmark

We sample a handful of gold (question, answer) pairs, run the pipeline, and
score generated answers against the gold answers with a simple Arabic
token-F1 (fast, no extra model) and, optionally, BERTScore (slower, needs a
BERT model download, but is the metric the paper found correlates best with
human judgment).


In [ ]:
import re

def normalize_ar(text):
    text = re.sub(r"[\u064B-\u0652]", "", text)  # strip diacritics
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

def token_f1(pred, gold):
    pred_toks, gold_toks = normalize_ar(pred), normalize_ar(gold)
    if not pred_toks or not gold_toks:
        return 0.0
    common = set(pred_toks) & set(gold_toks)
    if not common:
        return 0.0
    precision = len(common) / len(pred_toks)
    recall = len(common) / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

eval_sample = qa_df_clean.sample(n=10, random_state=0)[["Question", "Answer"]].reset_index(drop=True)

preds = []
for q in eval_sample["Question"]:
    ans, _ = answer_question(q, verbose=False)
    preds.append(ans)
eval_sample["Predicted_Answer"] = preds
eval_sample["token_f1"] = [
    token_f1(p, g) for p, g in zip(eval_sample["Predicted_Answer"], eval_sample["Answer"])
]

print("Mean token-F1:", eval_sample["token_f1"].mean())
eval_sample


In [ ]:
# Optional: BERTScore (slower — downloads a multilingual BERT model)
from bert_score import score as bert_score

P, R, F1 = bert_score(
    eval_sample["Predicted_Answer"].tolist(),
    eval_sample["Answer"].tolist(),
    lang="ar",
    verbose=True,
)
eval_sample["bertscore_recall"] = R.numpy()
eval_sample["bertscore_f1"] = F1.numpy()
print("Mean BERTScore recall:", eval_sample["bertscore_recall"].mean())
print("Mean BERTScore F1:", eval_sample["bertscore_f1"].mean())
eval_sample


## Notes / next steps

- **Scale up the corpus**: the 15 full Tafsir books (`Resources/Tafaser/Tafaser_DS1`,
  `Tafaser_DS2`, `.xlsx` files) contain far more verse-by-verse commentary than
  what's embedded in the QA dataset alone. Download and merge them into
  `chunks` the same way for a larger retrieval index.
- **Ontology-aware retrieval**: this notebook retrieves purely by embedding
  similarity. A closer match to the paper would also filter/boost candidates
  by matching the question's predicted ontology category
  (`Q_Type1` / `Q_type2_final_English`) against each chunk's `ontology_types`.
- **Swap in GPT-4** by adding an `OPENAI_API_KEY` Kaggle secret and changing
  `model="gpt-4o-mini"` to `"gpt-4o"` in the generation cell, to get closer to
  the paper's best-reported results.
- **Full benchmark run**: replace `qa_df.sample(n=10, ...)` with the full
  `tafsir-questions.csv` benchmark for a complete evaluation (will take much
  longer with the free local LLM).
